# **Multiple Regression Models to Predict Financial Returns**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.linear_model import LinearRegression

In [3]:
company=["HINDCOPPER.NS","HINDZINC.NS","DIXON.NS","JBMA.NS"]
data=yf.download(company,period="2y",interval="4h")
df=pd.DataFrame(data["Close"])
df

/tmp/ipython-input-3799512013.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data=yf.download(company,period="2y",interval="4h")
[*********************100%***********************]  4 of 4 completed


Ticker,DIXON.NS,HINDCOPPER.NS,HINDZINC.NS,JBMA.NS
Datetime,,,,
2024-01-15 03:45:00+00:00,6374.950195,258.850006,323.149994,914.500000
2024-01-15 07:45:00+00:00,6331.000000,256.950012,322.100006,920.000000
2024-01-16 03:45:00+00:00,6321.750000,262.399994,318.350006,921.900024
2024-01-16 07:45:00+00:00,6341.000000,264.250000,320.950012,934.200012
2024-01-17 03:45:00+00:00,6376.799805,263.399994,315.250000,941.000000
...,...,...,...,...
2026-01-12 07:45:00+00:00,11839.000000,547.200012,629.950012,604.000000
2026-01-13 03:45:00+00:00,11242.000000,541.500000,626.900024,604.049988
2026-01-13 07:45:00+00:00,11247.000000,539.349976,631.000000,607.700012


**Finding `lag 1` to `lag 5` for all companies**

In [4]:
lag_cols=pd.DataFrame(index=df.index)    # stores series of lag for each company
LR_cols=pd.DataFrame(index=df.index)    # stores series of LR for each company
lag_names=[]    # stores the column name of lag for each company
LR_names=[]    # stores the column name of LR for each company
for ticker in company:
  df[f"LR {ticker}"]=np.log(df[ticker]/df[ticker].shift(1))
  LR_cols[f"LR {ticker}"]=df[f"LR {ticker}"]
  LR_names.append(f"LR {ticker}")
  for lag in range(1,6):
    df[f"Lag {lag} of {ticker}"]=df[f"LR {ticker}"].shift(lag)
    lag_cols[f"Lag {lag} of {ticker}"]=df[f"Lag {lag} of {ticker}"]
    lag_names.append(f"Lag {lag} of {ticker}")
lag_cols.dropna(inplace=True)
LR_cols.dropna(inplace=True)
df.dropna(inplace=True);df

Ticker,DIXON.NS,HINDCOPPER.NS,HINDZINC.NS,JBMA.NS,LR HINDCOPPER.NS,Lag 1 of HINDCOPPER.NS,Lag 2 of HINDCOPPER.NS,Lag 3 of HINDCOPPER.NS,Lag 4 of HINDCOPPER.NS,Lag 5 of HINDCOPPER.NS,...,Lag 2 of DIXON.NS,Lag 3 of DIXON.NS,Lag 4 of DIXON.NS,Lag 5 of DIXON.NS,LR JBMA.NS,Lag 1 of JBMA.NS,Lag 2 of JBMA.NS,Lag 3 of JBMA.NS,Lag 4 of JBMA.NS,Lag 5 of JBMA.NS
Datetime,,,,,,,,,,,,,,,,,,,,,
2024-01-18 03:45:00+00:00,6296.450195,257.000000,311.250000,944.025024,-0.014678,-0.009920,-0.003222,0.007026,0.020988,-0.007367,...,0.005630,0.003040,-0.001462,-0.006918,-0.016781,0.019990,0.007253,0.013254,0.002063,0.005996
2024-01-18 07:45:00+00:00,6290.000000,253.649994,313.250000,943.424988,-0.013121,-0.014678,-0.009920,-0.003222,0.007026,0.020988,...,-0.001428,0.005630,0.003040,-0.001462,-0.000636,-0.016781,0.019990,0.007253,0.013254,0.002063
2024-01-19 03:45:00+00:00,6149.899902,264.350006,317.299988,953.974976,0.041319,-0.013121,-0.014678,-0.009920,-0.003222,0.007026,...,-0.011252,-0.001428,0.005630,0.003040,0.011121,-0.000636,-0.016781,0.019990,0.007253,0.013254
2024-01-19 07:45:00+00:00,6102.000000,265.750000,314.399994,951.000000,0.005282,0.041319,-0.013121,-0.014678,-0.009920,-0.003222,...,-0.001025,-0.011252,-0.001428,0.005630,-0.003123,0.011121,-0.000636,-0.016781,0.019990,0.007253
2024-01-23 03:45:00+00:00,5970.000000,263.899994,309.850006,914.500000,-0.006986,0.005282,0.041319,-0.013121,-0.014678,-0.009920,...,-0.022525,-0.001025,-0.011252,-0.001428,-0.039137,-0.003123,0.011121,-0.000636,-0.016781,0.019990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-12 07:45:00+00:00,11839.000000,547.200012,629.950012,604.000000,0.011025,0.038423,-0.002302,0.000958,-0.011250,-0.043231,...,0.000504,-0.007036,-0.001085,0.017241,0.013585,-0.029680,-0.019282,-0.016562,-0.002825,-0.033068
2026-01-13 03:45:00+00:00,11242.000000,541.500000,626.900024,604.049988,-0.010471,0.011025,0.038423,-0.002302,0.000958,-0.011250,...,-0.004379,0.000504,-0.007036,-0.001085,0.000083,0.013585,-0.029680,-0.019282,-0.016562,-0.002825
2026-01-13 07:45:00+00:00,11247.000000,539.349976,631.000000,607.700012,-0.003978,-0.010471,0.011025,0.038423,-0.002302,0.000958,...,-0.000929,-0.004379,0.000504,-0.007036,0.006024,0.000083,0.013585,-0.029680,-0.019282,-0.016562


In [5]:
LR_names

['LR HINDCOPPER.NS', 'LR HINDZINC.NS', 'LR DIXON.NS', 'LR JBMA.NS']

In [6]:
LM=LinearRegression(fit_intercept=True)
LM.fit(df[lag_names],df[LR_names])

LinearRegression()

In [7]:
LM.coef_

array([[ 0.01347922,  0.0332478 , -0.02330409,  0.05263812, -0.0187032 ,
         0.02638436,  0.01305559,  0.0079876 , -0.02436942,  0.038803  ,
         0.02593973, -0.12823257, -0.04765526, -0.05293162, -0.04870371,
         0.02548812, -0.01942549,  0.01256219,  0.04558116,  0.01496528],
       [-0.01964071, -0.02433116, -0.00723535,  0.03913345, -0.0167817 ,
         0.0886684 ,  0.03940306, -0.01646228,  0.01403442,  0.09640513,
        -0.01360463,  0.02330426,  0.01938867,  0.0078927 , -0.01133012,
         0.01039816, -0.03618395, -0.03930231, -0.02004524,  0.00951658],
       [-0.00424717, -0.04412395,  0.02022389, -0.04675288, -0.00025463,
        -0.02566421,  0.01015421, -0.00711751, -0.02293881,  0.00339929,
         0.05474698,  0.01664702, -0.01825921,  0.01319743, -0.02756673,
         0.04747486, -0.01148568, -0.01405391,  0.02822218,  0.03553292],
       [ 0.03567811,  0.020953  , -0.02146673, -0.00760771,  0.03535529,
         0.01878575,  0.0512882 , -0.00245247,  

In [8]:
LM.intercept_

array([ 0.00089607,  0.00055962,  0.00067237, -0.00066081])

In [9]:
predict_cols=pd.DataFrame(index=df.index)
predict_names=[]

# Make predictions using the entire feature set (all lag_names)
# The output will be a 2D array where each column corresponds to a target in LR_names
predictions = LM.predict(df[lag_names])
predictions=np.sign(predictions)

# Create a DataFrame from the predictions with appropriate column names
predicted_df = pd.DataFrame(predictions, index=df.index, columns=[f"Predicted {lr_name}" for lr_name in LR_names])

# Add these predicted columns to the main df and the predict_cols DataFrame
for lr_name in LR_names:
  col_name = f"Predicted {lr_name}"
  df[col_name] = predicted_df[col_name]
  predict_cols[col_name] = predicted_df[col_name]
  predict_names.append(col_name)
df

Ticker,DIXON.NS,HINDCOPPER.NS,HINDZINC.NS,JBMA.NS,LR HINDCOPPER.NS,Lag 1 of HINDCOPPER.NS,Lag 2 of HINDCOPPER.NS,Lag 3 of HINDCOPPER.NS,Lag 4 of HINDCOPPER.NS,Lag 5 of HINDCOPPER.NS,...,LR JBMA.NS,Lag 1 of JBMA.NS,Lag 2 of JBMA.NS,Lag 3 of JBMA.NS,Lag 4 of JBMA.NS,Lag 5 of JBMA.NS,Predicted LR HINDCOPPER.NS,Predicted LR HINDZINC.NS,Predicted LR DIXON.NS,Predicted LR JBMA.NS
Datetime,,,,,,,,,,,,,,,,,,,,,
2024-01-18 03:45:00+00:00,6296.450195,257.000000,311.250000,944.025024,-0.014678,-0.009920,-0.003222,0.007026,0.020988,-0.007367,...,-0.016781,0.019990,0.007253,0.013254,0.002063,0.005996,1.0,1.0,1.0,-1.0
2024-01-18 07:45:00+00:00,6290.000000,253.649994,313.250000,943.424988,-0.013121,-0.014678,-0.009920,-0.003222,0.007026,0.020988,...,-0.000636,-0.016781,0.019990,0.007253,0.013254,0.002063,-1.0,-1.0,-1.0,-1.0
2024-01-19 03:45:00+00:00,6149.899902,264.350006,317.299988,953.974976,0.041319,-0.013121,-0.014678,-0.009920,-0.003222,0.007026,...,0.011121,-0.000636,-0.016781,0.019990,0.007253,0.013254,1.0,1.0,1.0,-1.0
2024-01-19 07:45:00+00:00,6102.000000,265.750000,314.399994,951.000000,0.005282,0.041319,-0.013121,-0.014678,-0.009920,-0.003222,...,-0.003123,0.011121,-0.000636,-0.016781,0.019990,0.007253,1.0,1.0,1.0,1.0
2024-01-23 03:45:00+00:00,5970.000000,263.899994,309.850006,914.500000,-0.006986,0.005282,0.041319,-0.013121,-0.014678,-0.009920,...,-0.039137,-0.003123,0.011121,-0.000636,-0.016781,0.019990,1.0,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-12 07:45:00+00:00,11839.000000,547.200012,629.950012,604.000000,0.011025,0.038423,-0.002302,0.000958,-0.011250,-0.043231,...,0.013585,-0.029680,-0.019282,-0.016562,-0.002825,-0.033068,-1.0,-1.0,-1.0,1.0
2026-01-13 03:45:00+00:00,11242.000000,541.500000,626.900024,604.049988,-0.010471,0.011025,0.038423,-0.002302,0.000958,-0.011250,...,0.000083,0.013585,-0.029680,-0.019282,-0.016562,-0.002825,1.0,1.0,-1.0,1.0
2026-01-13 07:45:00+00:00,11247.000000,539.349976,631.000000,607.700012,-0.003978,-0.010471,0.011025,0.038423,-0.002302,0.000958,...,0.006024,0.000083,0.013585,-0.029680,-0.019282,-0.016562,-1.0,1.0,-1.0,-1.0


In [10]:
df['Predicted LR HINDCOPPER.NS'].value_counts()

,count
Predicted LR HINDCOPPER.NS,
1.0,574
-1.0,405
